# GT 기반 대학교 약어(연대, 건대 등) 규칙 생성 + 충돌 탐지

규칙: `~대학교` 이름에서 "학교"를 떼고 첫 글자 + "대"를 약어로 본다 (예: 연세대학교 -> 연대).
같은 약어에 서로 다른 정식명이 몰리는 충돌부터 찾고, 그중 실제로 우리 댓글에 등장하는 것만 골라서
수동으로 처리할 대상을 좁힌다.

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("repo 루트를 못 찾았어요 (data/, src/ 폴더 기준)")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.append(str(REPO_ROOT / "src"))

REPO_ROOT

In [ ]:
import pandas as pd

gt_df = pd.read_csv(REPO_ROOT / "data" / "processed" / "gt_schoolnames.csv", encoding="utf-8-sig")
gt_df.shape

## 1. 규칙 기반 약어 생성 — `~대학교`에서 "학교" 떼고 첫 글자 + "대"

In [ ]:
def generate_abbreviation(name: str) -> str | None:
    """'~대학교' 이름에서만 규칙 적용. 그 외(전문대/기능대 등 '~대학'류)는 규칙이 안 맞아서 제외."""
    if not name.endswith("대학교"):
        return None
    stem = name[:-2]  # "학교" 제거 (예: 연세대학교 -> 연세대)
    if len(stem) < 2:
        return None
    return stem[0] + "대"


abbrev_df = gt_df.assign(약어=gt_df["학교명"].apply(generate_abbreviation)).dropna(subset=["약어"])
abbrev_df = abbrev_df.rename(columns={"학교명": "정식명"})[["약어", "정식명"]].reset_index(drop=True)
abbrev_df.shape

## 2. 충돌 탐지 — 같은 약어에 서로 다른 정식명이 여러 개 걸리는 경우

In [ ]:
counts = abbrev_df.groupby("약어")["정식명"].nunique()
collision_keys = counts[counts > 1].index

collisions_df = abbrev_df[abbrev_df["약어"].isin(collision_keys)].sort_values("약어")
print(f"충돌나는 약어 {len(collision_keys)}개, 관련 정식명 {len(collisions_df)}개")
collisions_df

In [ ]:
# 충돌 없는(약어 하나 = 정식명 하나) 것들만 1차 후보로 채택
safe_df = abbrev_df[~abbrev_df["약어"].isin(collision_keys)].reset_index(drop=True)
safe_df.shape

## 3. 실제 우리 댓글의 미매칭 후보와 대조 (`gt_match_results.csv` 필요 — `gt_match.ipynb` 먼저 실행)

In [ ]:
results_df = pd.read_csv(
    REPO_ROOT / "data" / "processed" / "gt_match_results.csv", encoding="utf-8-sig"
)
results_df["school_candidate"] = results_df["school_candidate"].fillna("")

mismatch_df = results_df[
    (results_df["school_candidate_count"] >= 1) & (results_df["gt_match_count"] == 0)
]
our_tokens = sorted(
    {tok for candidates in mismatch_df["school_candidate"].str.split() for tok in candidates}
)
print(f"우리 데이터의 미매칭 고유 토큰 {len(our_tokens)}개")
our_tokens

In [ ]:
# 충돌 없이 자동 생성된 약어 중, 실제 우리 데이터에 등장하는 것만 최종 후보로 채택
auto_candidates = safe_df[safe_df["약어"].isin(our_tokens)]
print(f"자동 채택 가능한 약어 {len(auto_candidates)}개")
auto_candidates

In [ ]:
# 충돌나는 약어인데 우리 데이터에도 등장하는 것 -> 이것만 수동으로 확인 필요
needs_manual = collisions_df[collisions_df["약어"].isin(our_tokens)]
print(f"수동 확인 필요 {len(needs_manual)}개 (약어 {needs_manual['약어'].nunique()}개)")
needs_manual

`needs_manual`에 뜬 것들은 자동으로 못 정함 — `gt_schoolnames.csv`를 열어서 해당 정식명 행의 `약어` 컬럼에 직접 채워넣어야 함. 아래 셀은 충돌 없는(`auto_candidates`) 것들만 자동으로 채운다.

In [ ]:
# 별도 CSV가 아니라 gt_schoolnames.csv 자체에 "약어" 컬럼으로 병합해서 덮어씀 (재현 가능:
# build_school_dict.ipynb -> gt_match.ipynb -> 이 노트북 순서로 다시 돌리면 동일하게 재생성됨)
alias_map = dict(zip(auto_candidates["정식명"], auto_candidates["약어"]))
gt_df["약어"] = gt_df["학교명"].map(alias_map)

out_path = REPO_ROOT / "data" / "processed" / "gt_schoolnames.csv"
gt_df.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"약어 채워진 행: {gt_df['약어'].notna().sum()}개")
out_path